# Улучшенный пример загрузки данных и визуализации

Этот ноутбук демонстрирует загрузку одной записи из предобработанных данных, их визуализацию с разметкой приступов и сохранение графиков локально.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import os

# Настройка стиля графиков
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Создание директории для сохранения графиков
PLOTS_DIR = Path("../plots")
PLOTS_DIR.mkdir(exist_ok=True)

print(f"Графики будут сохранены в: {PLOTS_DIR.resolve()}")

Графики будут сохранены в: D:\NIR\Epilepsy\plots


In [ ]:
def load_single_processed_recording(animal_id, session_id, data_dir="../test_output_small"):
    """
    Загрузка одной записи из предобработанных данных
    
    Параметры:
    animal_id (str): идентификатор животного
    session_id (str): идентификатор сессии
    data_dir (str): путь к директории с обработанными данными
    
    Возвращает:
    tuple: (сигналы, сегменты, метаданные, маска приступов)
    """
    session_path = Path(data_dir) / animal_id / session_id
    
    if not session_path.exists():
        raise FileNotFoundError(f"Сессия {animal_id}/{session_id} не найдена")
    
    # Загрузка обработанных сигналов
    signals_file = session_path / "processed_signals.npy"
    if not signals_file.exists():
        raise FileNotFoundError(f"Файл сигналов не найден: {signals_file}")
    signals = np.load(signals_file)
    
    # Загрузка информации о сегментах
    segments_file = session_path / "segments_info.csv"
    if not segments_file.exists():
        raise FileNotFoundError(f"Файл сегментов не найден: {segments_file}")
    segments_df = pd.read_csv(segments_file)
    
    # Загрузка маски приступов
    seizure_mask_file = session_path / "seizure_mask.npy"
    if not seizure_mask_file.exists():
        raise FileNotFoundError(f"Файл маски приступов не найден: {seizure_mask_file}")
    seizure_mask = np.load(seizure_mask_file)
    
    # Загрузка метаданных
    metadata_file = session_path / "conversion_metadata.json"
    if metadata_file.exists():
        with open(metadata_file, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
    else:
        metadata = {}
    
    return signals, segments_df, metadata, seizure_mask

SyntaxError: unterminated string literal (detected at line 33) (2507560088.py, line 33)

In [ ]:
# Загрузка одной записи
try:
    animal_id = "Dex1x2NE"
    session_id = "BL_24Jan"
    
    print(f"Загрузка данных для животного {animal_id}, сессия {session_id}")
    signals, segments_df, metadata, seizure_mask = load_single_processed_recording(animal_id, session_id)
    
    print(f"Формат сигналов: {signals.shape}")
    print(f"Количество сегментов: {len(segments_df)}")
    print(f"Формат маски приступов: {seizure_mask.shape}")
    print(f"Частота дискретизации: {metadata.get('sampling_freq', 'Не указана')} Гц")
    print(f"Каналы: {metadata.get('channel_names', 'Не указаны')}")
    
except Exception as e:
    print(f"Ошибка при загрузке данных: {e}")

In [ ]:
def plot_individual_channels_with_seizures(signals, segments_df, metadata, channel_names=None, n_channels_to_plot=3):
    """
    Визуализация отдельных каналов с разметкой приступов
    
    Параметры:
    signals (np.ndarray): массив сигналов
    segments_df (pd.DataFrame): DataFrame с информацией о сегментах
    metadata (dict): метаданные записи
    channel_names (list): имена каналов (если None, используются из метаданных)
    n_channels_to_plot (int): количество каналов для отображения
    """
    sampling_freq = metadata.get('sampling_freq', 400.0)
    
    if channel_names is None:
        channel_names = metadata.get('channel_names', [f"Channel_{i}" for i in range(signals.shape[0])])
    
    # Ограничиваем количество каналов
    n_channels = min(n_channels_to_plot, signals.shape[0])
    
    # Получаем сегменты с приступами
    seizure_segments = segments_df[segments_df['segment_type'] == 'seizure']
    
    if len(seizure_segments) == 0:
        print("Не найдено сегментов с приступами для визуализации")
        return
    
    # Выбираем первый сегмент с приступом для визуализации
    seizure_segment = seizure_segments.iloc[0]
    start_sample = int(seizure_segment['start_sample'])
    end_sample = int(seizure_segment['end_sample'])
    
    # Расширяем диапазон для лучшей визуализации
    margin_samples = int(2 * sampling_freq)  # 2 секунды до и после
    start_sample = max(0, start_sample - margin_samples)
    end_sample = min(signals.shape[1], end_sample + margin_samples)
    
    # Создаем графики для разных каналов
    fig, axes = plt.subplots(n_channels, 1, figsize=(15, 3*n_channels))
    if n_channels == 1:
        axes = [axes]
    
    time_axis = np.arange(start_sample, end_sample) / sampling_freq
    
    for ch_idx in range(n_channels):
        window_data = signals[ch_idx, start_sample:end_sample]
        axes[ch_idx].plot(time_axis, window_data, linewidth=1, color='blue')
        axes[ch_idx].set_title(f"Канал {ch_idx}: {channel_names[ch_idx]}")
        axes[ch_idx].set_xlabel("Время (сек)")
        axes[ch_idx].set_ylabel("Амплитуда (мкВ)")
        axes[ch_idx].grid(True, alpha=0.3)
        
        # Выделение областей приступов
        for _, segment in seizure_segments.iterrows():
            seg_start = segment['start_sample'] / sampling_freq
            seg_end = segment['end_sample'] / sampling_freq
            
            # Проверяем пересечение с текущим окном
            if seg_start <= time_axis[-1] and seg_end >= time_axis[0]:
                # Ограничиваем пределы окном
                display_start = max(seg_start, time_axis[0])
                display_end = min(seg_end, time_axis[-1])
                
                axes[ch_idx].axvspan(display_start, display_end, alpha=0.3, color='red', 
                                   label='Приступ' if ch_idx == 0 else "")
        
        if ch_idx == 0:  # Добавляем легенду только на первый график
            axes[ch_idx].legend()
    
    plt.suptitle(f"Сигналы с разметкой приступов для {animal_id}/{session_id}")
    plt.tight_layout()
    
    # Сохраняем график
    plot_filename = PLOTS_DIR / f"{animal_id}_{session_id}_individual_channels.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"График сохранен: {plot_filename}")
    
    plt.show()

In [ ]:
# Визуализация отдельных каналов с разметкой приступов
if 'signals' in locals() and 'segments_df' in locals() and 'metadata' in locals():
    print("\nВизуализация отдельных каналов с разметкой приступов:")
    plot_individual_channels_with_seizures(signals, segments_df, metadata)

In [ ]:
def plot_overview_with_seizure_markers(signals, segments_df, metadata, max_duration=300):
    """
    Общая визуализация всех каналов для участков с приступами
    
    Параметры:
    signals (np.ndarray): массив сигналов
    segments_df (pd.DataFrame): DataFrame с информацией о сегментах
    metadata (dict): метаданные записи
    max_duration (float): максимальная длительность для отображения (в секундах)
    """
    sampling_freq = metadata.get('sampling_freq', 400.0)
    channel_names = metadata.get('channel_names', [f"Channel_{i}" for i in range(signals.shape[0])])
    
    # Ограничиваем длительность для лучшей визуализации
    max_samples = int(max_duration * sampling_freq)
    
    # Если сигнал длиннее максимальной длительности, берем только начало
    if signals.shape[1] > max_samples:
        signals_to_plot = signals[:, :max_samples]
        time_axis = np.arange(max_samples) / sampling_freq
    else:
        signals_to_plot = signals
        time_axis = np.arange(signals.shape[1]) / sampling_freq
    
    # Нормализация сигналов для лучшей визуализации
    normalized_signals = signals_to_plot.copy()
    for ch in range(normalized_signals.shape[0]):
        ch_mean = np.mean(normalized_signals[ch, :])
        ch_std = np.std(normalized_signals[ch, :])
        if ch_std > 0:
            normalized_signals[ch, :] = (normalized_signals[ch, :] - ch_mean) / ch_std
    
    # Создание графика
    fig, ax = plt.subplots(figsize=(20, 8))
    
    # Отображение сигналов с вертикальным смещением
    for ch in range(normalized_signals.shape[0]):
        ax.plot(time_axis, normalized_signals[ch, :] + ch * 3, linewidth=0.8, 
                label=channel_names[ch])
    
    # Выделение областей приступов
    seizure_segments = segments_df[segments_df['segment_type'] == 'seizure']
    for _, segment in seizure_segments.iterrows():
        seg_start = segment['start_time']
        seg_end = segment['end_time']
        
        # Ограничиваем пределы графиком
        display_start = max(0, seg_start)
        display_end = min(time_axis[-1], seg_end)
        
        if display_start < time_axis[-1] and display_end > 0:
            ax.axvspan(display_start, display_end, alpha=0.3, color='red')
    
    ax.set_title(f"Обзорная визуализация сигналов с разметкой приступов для {animal_id}/{session_id}")
    ax.set_xlabel("Время (сек)")
    ax.set_ylabel("Нормализованная амплитуда (условные единицы)")
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Сохраняем график
    plot_filename = PLOTS_DIR / f"{animal_id}_{session_id}_overview.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"График сохранен: {plot_filename}")
    
    plt.show()

In [ ]:
# Общая визуализация всех каналов для участков с приступами
if 'signals' in locals() and 'segments_df' in locals() and 'metadata' in locals():
    print("\nОбщая визуализация всех каналов для участков с приступами:")
    plot_overview_with_seizure_markers(signals, segments_df, metadata)

In [ ]:
def plot_seizure_statistics(segments_df):
    """
    Визуализация статистики приступов
    
    Параметры:
    segments_df (pd.DataFrame): DataFrame с информацией о сегментах
    """
    # Общая статистика
    seizure_segments = segments_df[segments_df['segment_type'] == 'seizure']
    normal_segments = segments_df[segments_df['segment_type'] == 'normal']
    
    total_seizure_duration = seizure_segments['duration'].sum()
    total_normal_duration = normal_segments['duration'].sum()
    total_duration = total_seizure_duration + total_normal_duration
    
    print(f"\nСтатистика приступов:")
    print(f"Количество приступов: {len(seizure_segments)}")
    print(f"Общая длительность приступов: {total_seizure_duration:.2f} сек")
    print(f"Общая длительность записи: {total_duration:.2f} сек")
    print(f"Процент времени с приступами: {total_seizure_duration/total_duration*100:.2f}%")
    
    # Создание графиков
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Гистограмма типов сегментов
    segment_counts = segments_df['segment_type'].value_counts()
    colors = ['skyblue' if x == 'normal' else 'salmon' for x in segment_counts.index]
    ax1.bar(segment_counts.index, segment_counts.values, color=colors)
    ax1.set_title('Распределение типов сегментов')
    ax1.set_ylabel('Количество сегментов')
    
    # Длительности приступов
    if len(seizure_segments) > 0:
        ax2.hist(seizure_segments['duration'], bins=10, color='salmon', alpha=0.7, edgecolor='black')
        ax2.set_title('Распределение длительностей приступов')
        ax2.set_xlabel('Длительность (сек)')
        ax2.set_ylabel('Количество приступов')
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, 'Нет приступов для анализа', 
                horizontalalignment='center', verticalalignment='center',
                transform=ax2.transAxes)
        ax2.set_title('Распределение длительностей приступов')
    
    plt.tight_layout()
    
    # Сохраняем график
    plot_filename = PLOTS_DIR / f"{animal_id}_{session_id}_statistics.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"График сохранен: {plot_filename}")
    
    plt.show()

In [ ]:
# Визуализация статистики приступов
if 'segments_df' in locals():
    print("\nСтатистика приступов:")
    plot_seizure_statistics(segments_df)

## Заключение

Этот ноутбук демонстрирует:
1. Загрузку одной записи из предобработанных данных
2. Визуализацию отдельных каналов с разметкой приступов
3. Общую визуализацию всех каналов для участков с приступами
4. Статистику приступов
5. Сохранение всех графиков в локальную директорию plots/

Все графики сохраняются в высоком разрешении (300 DPI) для последующего использования.